# VLM-based Document Augmentation

This notebook extracts structured features (weight, color, size, capacity, etc.) from product **name, description AND image** using a Vision Language Model (VLM) and saves an augmented CSV for later indexing.

The VLM sees the product image alongside text, so it can extract visual attributes (color, shape, material) that text alone might miss.

Output columns:
- `features_json`: structured attributes as JSON
- `augmented_text`: original text + normalized features


In [ ]:
from pathlib import Path
import json
import pandas as pd
import torch
from tqdm import tqdm
from PIL import Image
from transformers import pipeline

PRODUCTS_CSV = Path("dataset/products.csv")
IMAGES_DIR = Path("dataset/images")
OUTPUT_CSV = Path("dataset/products_augmented.csv")

# VLM that can see images — change to "Qwen/Qwen2.5-VL-3B-Instruct" if needed
VLM_MODEL = "Qwen/Qwen3-VL-4B-Instruct"
MAX_NEW_TOKENS = 8192


In [ ]:
pipe = pipeline(
    "image-text-to-text",
    model=VLM_MODEL,
    dtype=torch.float16,
    device_map="auto",
    model_kwargs={"attn_implementation": "eager"},  # Windows — no Triton
)
print(f"VLM pipeline ready: {VLM_MODEL}")

In [ ]:
products_df = pd.read_csv(PRODUCTS_CSV)
print(f"Loaded {len(products_df)} products")
print(products_df[["db_id", "name", "price", "image"]].head())

In [ ]:
SYSTEM_PROMPT = (
    "You are a product feature extractor. Given a product's name, description, price "
    "and image, extract ALL structured attributes you can identify from BOTH the text "
    "AND the image. Return ONLY valid JSON with these keys: "
    "product_name, brand, model, color, size, weight, capacity, dimensions, material, "
    "price, features (list), other_attributes (dict). ",
    "IMPORTANT: ALWAYS GIVE YOUR DOCUMENT IN JSON FORMAT.",
    "Use empty string for unknown fields."
)

def ask_model(product_name, product_description, product_price, product_image_filename):
    """Extract structured features using the VLM pipeline (text + image)."""

    user_content = []

    # Embed image directly in message content (pipeline requires this for chat format)
    img_path = IMAGES_DIR / product_image_filename
    if img_path.exists():
        user_content.append({
            "type": "image",
            "image": Image.open(img_path).convert("RGB"),
        })

    user_content.append({
        "type": "text",
        "text": (
            f"Product Name: {product_name}\n"
            f"Product Description: {product_description}\n"
            f"Product Price: {product_price}\n"
            "Extract all product attributes from the text AND the image above."
        ),
    })

    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user",   "content": user_content},
    ]

    output = pipe(
        text=messages,
        max_length=MAX_NEW_TOKENS,
        return_full_text=False,
    )
    return output[0]["generated_text"]
    
# --- Quick test with a single product ---
features = ask_model(
    "Gevi White Espresso Machine 20-Bar New Latte Cappuccino Maker with Frother, 1.25 L",
    "Gevi Espresso Machines 20 Bar Fast Heating Automatic Cappuccino Coffee Maker "
    "with Foaming Milk Frother Wand for Espresso, Latte Macchiato, "
    "40.58 oz Removable Water Tank, 1350W, White",
    "139.99",
    "match_0_db_a957964f70be.jpeg",
)
print(features)

In [ ]:
augmented_list = []

for _, row in tqdm(products_df.iterrows(), total=len(products_df), desc="Augmenting"):
    name = str(row.get("name", ""))
    description = str(row.get("description", ""))
    price = str(row.get("price", "")) if pd.notna(row.get("price")) else ""
    image = str(row.get("image", ""))

    # Ask VLM to extract structured features from text + image
    raw_output = ask_model(name, description, price, image)

    # Build augmented text: original + extracted features
    base_text = f"{name} {description}".strip()
    augmented_text = f"{base_text}\nFeatures: {raw_output}"

    augmented_list.append({
        **row.to_dict(),
        "features_json": raw_output,
        "augmented_text": augmented_text,
    })

# Save to CSV
augmented_df = pd.DataFrame(augmented_list)
augmented_df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved {len(augmented_df)} augmented products to {OUTPUT_CSV}")
augmented_df[["db_id", "name", "features_json"]].head(3)


In [ ]:
# prepare the model input
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

In [ ]:
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=16384
)


In [ ]:
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

content = tokenizer.decode(output_ids, skip_special_tokens=True)

print(content)